# 02 Feature engineering
S1/S2/S3 및 중복 신호 점검. 미래 기록·label을 피처로 사용하지 않는다.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
if not (ROOT / 'data/battery_modeling_ready.parquet').exists():
    raise FileNotFoundError('프로젝트 ZIP을 풀고 프로젝트 루트 또는 notebooks 폴더에서 실행하세요.')
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image
from src.preprocess import load_data
df = load_data(ROOT / 'data/battery_modeling_ready.parquet')


In [ ]:
from src.features import FEATURE_SETS, derive_early_features, check_redundant_features
display(pd.DataFrame({'set':list(FEATURE_SETS), 'features':[' / '.join(v) for v in FEATURE_SETS.values()]}))
display(check_redundant_features(df))

In [ ]:
display(df[df.attrs['feature_columns']].isna().sum().to_frame('missing'))
print(df.attrs['definitions'])
assert set(df.loc[df.split.eq('train'),'group_id']).isdisjoint(df.loc[df.split.eq('valid'),'group_id'])
display(df.groupby('split').agg(cells=('battery_id','size'),label=('has_label','sum'),policies=('group_id','nunique')))

원 곡선에서 파생변수 생성은 `src.features.derive_early_features`에 구현했다. Q10>0 검사, 전압 순서 정렬, 분산 하한을 적용한다. 모든 전처리 fit은 `src.train` 내부 Pipeline의 학습 fold에 한정된다.